# Polymer Property Prediction — Exploratory Data Analysis

## Purpose

Understand the polymer dataset before cleaning, feature engineering
and predictive modelling.

## Research question

What information does the dataset contain, how reliable is it,
and which polymer properties can we model using the available labels?

## Data files

- train.csv
- test.csv
- sample_submission.csv
- dataset1.csv
- dataset2.csv
- dataset3.csv
- dataset4.csv

## Questions to investigate

1. What does one row represent?
2. How were the property values generated?
3. What are the units and simulation conditions?
4. Which targets are available for each structure?
5. Are structures duplicated within or across files?
6. Can we process the polymer SMILES consistently?
7. Are the supplementary datasets comparable to the main dataset?

## Analysis principles

- Preserve the original files.
- Do not impute missing target values.
- Investigate unusual values before removing them.
- Record sample sizes for every comparison.
- Separate observations, hypotheses and decisions.
- Keep the final evaluation set separate from model development.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("pandas version:", pd.__version__)
print("Working directory:", Path.cwd())

Python executable: c:\Users\ghsm0\Desktop\Bootcamp_DS_desktop\Capstone\capstone_virtual_lab_mohamed_suhmi\.venv\Scripts\python.exe
Python version: 3.13.14
pandas version: 3.0.5
Working directory: c:\Users\ghsm0\Desktop\Bootcamp_DS_desktop\Capstone\capstone_virtual_lab_mohamed_suhmi\notebooks


Checking the data folder. Our notebook operates within the `notebooks` subfolder. Therefore, we first move up one level to the project folder.

Pseudocode:

1- Determine the current working directory. (Aktuellen Arbeitsordner ermitteln.)

2- Identify the parent project folder. (Übergeordneten Projektordner bestimmen.)

3- Construct the path to the `data` folder. (Den Pfad zum Ordner data bilden.)

4- Check if this folder exists. (Prüfen, ob dieser Ordner existiert.)

5- List existing CSV files. (Vorhandene CSV-Dateien auflisten.)

In [8]:
# Locate the project and data directories
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Data directory exists:", DATA_DIR.is_dir())

# List CSV files, including files in subdirectories
csv_files = sorted(DATA_DIR.rglob("*.csv"))

print("Number of CSV files:", len(csv_files))

for file_path in csv_files:
    print(file_path.relative_to(PROJECT_ROOT))
    

Project root: c:\Users\ghsm0\Desktop\Bootcamp_DS_desktop\Capstone\capstone_virtual_lab_mohamed_suhmi
Data directory: c:\Users\ghsm0\Desktop\Bootcamp_DS_desktop\Capstone\capstone_virtual_lab_mohamed_suhmi\data
Data directory exists: True
Number of CSV files: 11
data\dataset1.csv
data\dataset2.csv
data\dataset3.csv
data\dataset4.csv
data\sample_submission.csv
data\test.csv
data\train.csv
data\train_supplement\dataset1.csv
data\train_supplement\dataset2.csv
data\train_supplement\dataset3.csv
data\train_supplement\dataset4.csv


## 2. Dataset inventory

Before analysing individual datasets, we inspect all seven CSV files.

For each file, we record:
- its relative path;
- the number of rows and columns;
- the column names.

This provides an overview of the available data and its structure.

| Bestandteil | Bedeutung | Meaning (English) |
|---|---|---|
| `datasets = {}` | Ein leeres Dictionary speichert die eingelesenen Tabellen. | An empty dictionary stores the loaded tables. |
| `inventory_rows = []` | Eine leere Liste sammelt die Angaben für unsere Übersicht. | An empty list collects the information for our overview. |
| `for file_path in csv_files:` | Bearbeitet jede der sieben gefundenen Dateien. | Processes each of the seven files found. |
| `.relative_to(DATA_DIR)` | Kürzt den Pfad auf den Teil innerhalb von `data`. | Makes the path relative to the `data` directory. |
| `.as_posix()` | Verwendet `/` im Pfad, damit die Bezeichnungen unter Windows und Linux gleich aussehen. | Uses `/` in the path so that the labels look the same on Windows and Linux. |
| `pd.read_csv(file_path)` | Liest die CSV-Datei als pandas-DataFrame ein. | Reads the CSV file into a pandas DataFrame. |
| `datasets[relative_path] = df` | Speichert die Tabelle unter ihrem Dateipfad für spätere Analysen. | Stores the table under its file path for later analysis. |
| `df.shape[0]` | Anzahl der Zeilen. | Number of rows. |
| `df.shape[1]` | Anzahl der Spalten. | Number of columns. |
| `", ".join(df.columns)` | Verbindet die Spaltennamen zu einem lesbaren Text. | Joins the column names into a readable string. |
| `inventory_rows.append({...})` | Fügt die Angaben einer Datei zur Übersicht hinzu. | Adds the information for one file to the overview. |
| `pd.DataFrame(inventory_rows)` | Wandelt die gesammelten Angaben in eine Tabelle um. | Converts the collected information into a table. |

In [9]:
# Load the CSV files and collect their basic structure
datasets = {}
inventory_rows = []

for file_path in csv_files:
    relative_path = file_path.relative_to(DATA_DIR).as_posix()
    df = pd.read_csv(file_path)

    datasets[relative_path] = df

    inventory_rows.append({
        "file": relative_path,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "column_names": ", ".join(df.columns),
    })

# Display the dataset inventory
inventory = pd.DataFrame(inventory_rows)
display(inventory)

,file,rows,columns,column_names
0,dataset1.csv,874,2,"SMILES, TC_mean"
1,dataset2.csv,7208,1,SMILES
2,dataset3.csv,46,2,"SMILES, Tg"
3,dataset4.csv,862,2,"SMILES, FFV"
4,sample_submission.csv,3,6,"id, Tg, FFV, Tc, Density, Rg"
5,test.csv,3,2,"id, SMILES"
6,train.csv,7973,7,"id, SMILES, Tg, FFV, Tc, Density, Rg"
7,train_supplement/dataset1.csv,874,2,"SMILES, TC_mean"
8,train_supplement/dataset2.csv,7208,1,SMILES
9,train_supplement/dataset3.csv,46,2,"SMILES, Tg"


## 3. Initial inspection of the training dataset

We inspect the first five rows, column data types and non-null counts.

Questions:
- Which columns contain identifiers, structures and prediction targets?
- Which data types did pandas infer?
- Which columns contain missing values?

At this stage, we do not modify or remove any observations.

## 3. Erste Untersuchung des Trainingsdatensatzes

Wir untersuchen die ersten fünf Zeilen, die Datentypen der Spalten sowie die Anzahl der Nicht-Null-Werte.

Fragen:
- Welche Spalten enthalten Identifikatoren, Strukturen und Zielvariablen?
- Welche Datentypen hat pandas abgeleitet?
- Welche Spalten enthalten fehlende Werte?

In dieser Phase verändern oder entfernen wir keine Beobachtungen.

In [14]:
# Select the training dataset
train = datasets.get("raw/train.csv")

if train is None:
    train = datasets.get("train.csv")

if train is None:
    raise KeyError("Training dataset not found in datasets.")

display(train.head())
train.info()
train.info()

,id,SMILES,Tg,FFV,Tc,Density,Rg
0,87817,*CC(*)c1ccccc1C(=O)OCCCCCC,NaN,0.374645,0.205667,NaN,NaN
1,106919,*Nc1ccc([C@H](CCC)c2ccc(C3(c4ccc([C@@H](CCC)c5...,NaN,0.370410,NaN,NaN,NaN
2,388772,*Oc1ccc(S(=O)(=O)c2ccc(Oc3ccc(C4(c5ccc(Oc6ccc(...,NaN,0.378860,NaN,NaN,NaN
3,519416,*Nc1ccc(-c2c(-c3ccc(C)cc3)c(-c3ccc(C)cc3)c(N*)...,NaN,0.387324,NaN,NaN,NaN
4,539187,*Oc1ccc(OC(=O)c2cc(OCCCCCCCCCOCC3CCCN3c3ccc([N...,NaN,0.355470,NaN,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 7973 entries, 0 to 7972
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   id       7973 non-null   int64  
 1   SMILES   7973 non-null   str    
 2   Tg       511 non-null    float64
 3   FFV      7030 non-null   float64
 4   Tc       737 non-null    float64
 5   Density  613 non-null    float64
 6   Rg       614 non-null    float64
dtypes: float64(5), int64(1), str(1)
memory usage: 436.2 KB
<class 'pandas.DataFrame'>
RangeIndex: 7973 entries, 0 to 7972
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   id       7973 non-null   int64  
 1   SMILES   7973 non-null   str    
 2   Tg       511 non-null    float64
 3   FFV      7030 non-null   float64
 4   Tc       737 non-null    float64
 5   Density  613 non-null    float64
 6   Rg       614 non-null    float64
dtypes: float64(5), int64(1), str(1)
memory usage: 436.2 KB


| Bestandteil | Bedeutung | Meaning (English) |
|---|---|---|
| `datasets` | Dictionary mit den bereits eingelesenen Tabellen. | Dictionary containing the tables already loaded. |
| `["raw/train.csv"]` | Wählt die Tabelle mit diesem Schlüssel aus dem Dictionary aus. | Selects the table with this key from the dictionary. |
| `train = ...` | Macht diese Tabelle unter dem Variablennamen `train` verfügbar. | Makes this table available under the variable name `train`. |
| `train.head()` | Gibt standardmäßig die ersten fünf Zeilen zurück. | Returns the first five rows by default. |
| `display(...)` | Zeigt das Ergebnis als Tabelle im Notebook an. | Displays the result as a table in the notebook. |
| `train.info()` | Zeigt Spaltennamen, Datentypen, Anzahl vorhandener Werte und Speicherverbrauch. | Shows column names, data types, non-null counts and memory usage. |

## 4. Missing-value analysis

We calculate the number and percentage of missing values in each column.

Missing target values are not replaced with zero or imputed.
Rows with partially available targets are preserved because they may
still be useful for predicting individual properties.

Pseudocode:
1. Count the available values in each column.
2. Count the missing values in each column.
3. Calculate the percentage of missing values.
4. Display the results in a summary table.

In [ ]:
# Summarise available and missing values for each column
missing_summary = pd.DataFrame({
    "available_count": train.notna().sum(),
    "missing_count": train.isna().sum(),
    "missing_percent": train.isna().mean() * 100,
})

# Round percentages for readability
missing_summary["missing_percent"] = (
    missing_summary["missing_percent"].round(2)
)

display(missing_summary)

,available_count,missing_count,missing_percent
id,7973,0,0.00
SMILES,7973,0,0.00
Tg,511,7462,93.59
FFV,7030,943,11.83
Tc,737,7236,90.76
Density,613,7360,92.31
Rg,614,7359,92.30


| Bestandteil | Bedeutung | Meaning (English) |
|---|---|---|
| `train.notna()` | Prüft jede Zelle: `True`, wenn ein Wert vorhanden ist. | Checks each cell: `True` if a value is available. |
| `train.isna()` | Prüft jede Zelle: `True`, wenn ein Wert fehlt. | Checks each cell: `True` if a value is missing. |
| `.sum()` | Zählt die `True`-Werte je Spalte. | Counts the `True` values in each column. |
| `.mean()` | Berechnet den Anteil der `True`-Werte je Spalte. | Calculates the proportion of `True` values in each column. |
| `* 100` | Wandelt den Anteil in Prozent um. | Converts the proportion into a percentage. |
| `pd.DataFrame({...})` | Erstellt eine Übersichtstabelle aus den Ergebnissen. | Creates a summary table from the results. |
| `.round(2)` | Rundet auf zwei Nachkommastellen. | Rounds to two decimal places. |
| `display(missing_summary)` | Zeigt die Übersicht im Notebook an. | Displays the summary in the notebook. |

## 5. Target availability per polymer

We count how many of the five target properties are available in each row.

This shows whether polymers have no labels, one label or multiple labels.
No observations are removed or modified.

Pseudocode:
1. Define the five target columns.
2. Check which target values are available.
3. Count the available targets in each row.
4. Count how many rows have 0, 1, 2, 3, 4 or 5 available targets.
5. Display counts and percentages.

In [ ]:
# Define the five prediction targets
target_columns = ["Tg", "FFV", "Tc", "Density", "Rg"]

# Count available targets in each row
available_targets_per_row = train[target_columns].notna().sum(axis=1)

# Count rows for each possible number of available targets
target_count_summary = (
    available_targets_per_row
    .value_counts()
    .reindex(range(6), fill_value=0)
    .sort_index()
    .rename_axis("available_targets")
    .to_frame(name="row_count")
)

# Calculate the percentage of rows in each group
target_count_summary["row_percent"] = (
    target_count_summary["row_count"] / len(train) * 100
).round(2)

display(target_count_summary)

,row_count,row_percent
available_targets,,
0,0,0.00
1,7266,91.13
2,122,1.53
3,345,4.33
4,240,3.01
5,0,0.00


| Bestandteil | Bedeutung | Meaning (English) |
|---|---|---|
| `available_targets_per_row` | Series mit der Anzahl vorhandener Zielwerte je Zeile. | Series containing the number of available targets per row. |
| `.value_counts()` | Zählt, wie häufig jede Anzahl vorkommt. | Counts how often each number occurs. |
| `range(6)` | Erzeugt die Zahlen 0 bis 5; die Obergrenze 6 ist ausgeschlossen. | Generates the numbers 0 through 5; the upper limit 6 is excluded. |
| `.reindex(...)` | Stellt sicher, dass alle sechs möglichen Anzahlen im Ergebnis stehen. | Ensures that all six possible counts appear in the result. |
| `fill_value=0` | Setzt die Häufigkeit nicht vorkommender Gruppen auf 0. Es ersetzt keine fehlenden Zielwerte. | Sets the frequency of absent groups to 0. It does not replace missing target values. |
| `.sort_index()` | Sortiert die Gruppen nach ihrer Anzahl von 0 bis 5. | Sorts the groups by their count from 0 to 5. |
| `.rename_axis("available_targets")` | Gibt dem Index der Übersicht einen Namen. | Names the index of the summary. |
| `.to_frame(name="row_count")` | Wandelt die Series in eine Tabelle mit der Spalte `row_count` um. | Converts the Series into a table with a column named `row_count`. |
| `len(train)` | Gibt die gesamte Anzahl der Trainingszeilen zurück. | Returns the total number of training rows. |
| `/ len(train) * 100` | Berechnet den Anteil jeder Gruppe in Prozent. | Calculates each group's percentage of all rows. |
| `.round(2)` | Rundet auf zwei Nachkommastellen. | Rounds to two decimal places. |

## 6. Pairwise target availability

We count how many rows contain both target values for each pair
of properties.

The diagonal contains the available-value count for each target.
The off-diagonal entries contain the number of shared labelled rows.

These counts are necessary for interpreting later correlations.
They do not measure correlation themselves.

Pseudocode:
1. Select the five target columns.
2. Mark available values as 1 and missing values as 0.
3. Count rows where both properties are available for each pair.
4. Display the pairwise counts.

In [ ]:
# Represent target availability as 1 (available) or 0 (missing)
target_availability = train[target_columns].notna().astype("int64")

# Count shared labelled rows for every pair of targets
pairwise_target_counts = target_availability.T.dot(target_availability)

display(pairwise_target_counts)

,Tg,FFV,Tc,Density,Rg
Tg,511,1,32,24,24
FFV,1,7030,300,270,270
Tc,32,300,737,531,535
Density,24,270,531,613,610
Rg,24,270,535,610,614


| Bestandteil | Bedeutung | Meaning (English) |
|---|---|---|
| `target_availability` | Variable für die Tabelle mit Verfügbarkeitsmarkierungen. | Variable containing the table of availability indicators. |
| `train[target_columns]` | Wählt die fünf Zielspalten aus. | Selects the five target columns. |
| `.notna()` | Markiert vorhandene Werte mit `True` und fehlende mit `False`. | Marks available values as `True` and missing values as `False`. |
| `.astype("int64")` | Wandelt Wahrheitswerte in Ganzzahlen um: `True` wird 1, `False` wird 0. | Converts Boolean values to integers: `True` becomes 1 and `False` becomes 0. |
| `"int64"` | Argument für den gewünschten Datentyp: Ganzzahlen mit 64 Bit. | Argument specifying the desired data type: 64-bit integers. |
| `.T` | Transponiert die Tabelle: Zeilen und Spalten tauschen ihre Rollen. Es ist ein Attribut und benötigt keine Aufrufklammern. | Transposes the table: rows and columns exchange roles. It is an attribute and requires no call parentheses. |
| `.dot(target_availability)` | Multipliziert die Tabellen als Matrizen. Hier ergibt das die Anzahl gemeinsam vorhandener Zielwerte für jedes Paar. | Performs matrix multiplication. Here, it produces the number of jointly available target values for each pair. |
| `pairwise_target_counts` | Variable für die resultierende Tabelle mit gemeinsamen Anzahlen. | Variable containing the resulting table of shared counts. |
| `display(...)` | Zeigt die Ergebnistabelle im Notebook an. | Displays the result table in the notebook. |


### Findings: pairwise target availability

- The diagonal matches the available-value counts for individual targets.
- Tg and FFV share only one labelled row, so their correlation cannot
  be calculated from this dataset.
- Tg shares only 24–32 labelled rows with the other targets.
  Any relationship analysis must account for these small sample sizes.
- Density and Rg share 610 labelled rows, indicating substantial overlap
  in target availability.
- Shared-label counts measure data availability, not correlation strength.
- The reasons for missing target values remain unknown.

### Ergebnisse: paarweise Zielgrößenverfügbarkeit

- Die Diagonale entspricht der Anzahl der verfügbaren Werte für die einzelnen Zielgrößen.
- Tg und FFV haben nur eine gemeinsame beschriftete Zeile, daher kann ihre Korrelation nicht
aus diesem Datensatz berechnet werden.
- Tg teilt nur 24–32 beschriftete Zeilen mit den anderen Zielen. 
Bei jeder Analyse von Zusammenhängen müssen diese geringen Stichprobenumfänge berücksichtigt werden.
- Dichte und Rg teilen sich 610 beschriftete Zeilen, was auf eine erhebliche Überschneidung hinweist.

in der Zielverfügbarkeit.
- Die Anzahl gemeinsamer Labels misst die Datenverfügbarkeit, nicht die Stärke der Korrelation.
- Die Gründe für das Fehlen von Werten für die Zielgrößen sind unbekannt.

In [ ]:
# Count duplicate occurrences after the first occurrence
duplicate_summary = pd.DataFrame({
    "check": [
        "Duplicate IDs",
        "Duplicate full rows",
        "Duplicate exact SMILES strings",
    ],
    "duplicate_count": [
        train["id"].duplicated().sum(),
        train.duplicated().sum(),
        train["SMILES"].duplicated().sum(),
    ],
})

display(duplicate_summary)

,check,duplicate_count
0,Duplicate IDs,0
1,Duplicate full rows,0
2,Duplicate exact SMILES strings,0


| Bestandteil | Bedeutung | Meaning (English) |
|---|---|---|
| `train["id"]` | Wählt die ID-Spalte als Series aus. | Selects the ID column as a Series. |
| `train["SMILES"]` | Wählt die SMILES-Spalte als Series aus. | Selects the SMILES column as a Series. |
| `.duplicated()` | Markiert Wiederholungen nach dem ersten Vorkommen mit `True`. Standard: `keep="first"`. | Marks repeated occurrences after the first as `True`. Default: `keep="first"`. |
| `train.duplicated()` | Prüft auf Zeilen, die in allen Spalten identisch sind. | Checks for rows that are identical across all columns. |
| `.sum()` | Zählt die als `True` markierten Wiederholungen. | Counts the repetitions marked as `True`. |
| `"check"` | Name der Spalte mit den Prüfbezeichnungen. | Name of the column containing the check labels. |
| `"duplicate_count"` | Name der Spalte mit den gezählten Wiederholungen. | Name of the column containing the repetition counts. |
| `pd.DataFrame({...})` | Erstellt die Übersicht aus einem Dictionary mit zwei Listen. | Creates the summary from a dictionary containing two lists. |
| `duplicate_summary` | Variable für die Ergebnistabelle. | Variable containing the result table. |
| `display(duplicate_summary)` | Zeigt die Prüfergebnisse an. | Displays the check results. |

### Findings: duplicate checks

- No duplicate IDs were found in the training dataset.
- No duplicate full rows were found.
- No duplicate exact SMILES strings were found.
- These checks compare stored values and text.
  Chemical structure validity and equivalence have not yet been checked.
- No rows were removed.

In [ ]:
# Select the SMILES column
smiles = train["SMILES"]

# Check missing values and whitespace
missing_smiles = smiles.isna()
blank_smiles = smiles.str.strip().eq("").fillna(False)
surrounding_whitespace = (
    smiles.ne(smiles.str.strip()).fillna(False)
)

# Display the results
smiles_text_summary = pd.DataFrame({
    "check": [
        "Missing SMILES",
        "Empty or whitespace-only SMILES",
        "SMILES with leading or trailing whitespace",
    ],
    "row_count": [
        missing_smiles.sum(),
        blank_smiles.sum(),
        surrounding_whitespace.sum(),
    ],
})

display(smiles_text_summary)

,check,row_count
0,Missing SMILES,0
1,Empty or whitespace-only SMILES,0
2,SMILES with leading or trailing whitespace,0


| Bestandteil | Bedeutung | Meaning (English) |
|---|---|---|
| `smiles` | Variable für die ausgewählte SMILES-Spalte. | Variable containing the selected SMILES column. |
| `.isna()` | Erkennt fehlende Werte. | Detects missing values. |
| `.str` | Zugriff auf Textmethoden einer pandas-Series. | Provides access to string methods of a pandas Series. |
| `.strip()` | Liefert Texte ohne Leerraum am Anfang oder Ende. Die Originalspalte bleibt unverändert. | Returns strings without leading or trailing whitespace. The original column remains unchanged. |
| `.eq("")` | Prüft, ob der Text nach dem Entfernen des äußeren Leerraums leer ist. | Checks whether the text is empty after removing surrounding whitespace. |
| `""` | Ein leerer String mit null Zeichen. | An empty string containing zero characters. |
| `.ne(...)` | Prüft auf Ungleichheit. Hier wird der Originaltext mit dem bereinigten Text verglichen. | Checks for inequality. Here, the original text is compared with the stripped text. |
| `.fillna(False)` | Behandelt fehlende Prüfergebnisse als `False`. Fehlende SMILES werden separat gezählt. | Treats missing check results as `False`. Missing SMILES are counted separately. |
| `.sum()` | Zählt die mit `True` markierten Zeilen. | Counts the rows marked as `True`. |
| `smiles_text_summary` | Variable für die Übersicht der Textprüfungen. | Variable containing the summary of text checks. |

### Findings: SMILES text quality

- All 7,973 training rows contain a SMILES entry.
- No empty or whitespace-only entries were found.
- No leading or trailing whitespace was found.
- No text cleaning was required for these checks.
- Chemical parsing and structure equivalence have not yet been checked.
- The original data remains unchanged.

In [ ]:
# Check whether RDKit is available in the notebook environment
import rdkit
from rdkit import Chem

print("RDKit version:", rdkit.__version__)

RDKit version: 2026.09.1


| Bestandteil | Bedeutung | Meaning (English) |
|---|---|---|
| `import` | Python-Schlüsselwort zum Importieren eines Moduls oder Pakets. | Python keyword for importing a module or package. |
| `rdkit` | Das Paket, das wir für die folgenden chemischen Strukturprüfungen verwenden möchten. | The package we intend to use for the following chemical structure checks. |
| `from rdkit import Chem` | Importiert das Modul `Chem` aus RDKit. | Imports the `Chem` module from RDKit. |
| `Chem` | Name, unter dem das importierte Modul anschließend verfügbar ist. | Name under which the imported module becomes available. |
| `rdkit.__version__` | Attribut mit der installierten RDKit-Version. | Attribute containing the installed RDKit version. |
| `print(...)` | Funktion, die ihre Argumente als Text ausgibt. | Function that prints its arguments as text. |
| `"RDKit version:"` | Text zur Beschriftung der Ausgabe. | Text labelling the output. |
| `,` | Trennt die beiden an `print()` übergebenen Argumente. | Separates the two arguments passed to `print()`. |

# Feature Engineering

## Convert SMILES strings to molecular objects and compute molecular weights

Workflow:

SMILES  
  ↓  
RDKit Mol  
  ↓  
Molecular Descriptors  
  ↓  
Numerical Feature Matrix  
  ↓  
Train/Test Split  
  ↓  
Random Forest / XGBoost  
  ↓  
Polymer Property Prediction  

### Step1: Convert SMILES into RDKit molecule

In [15]:
from rdkit import Chem

train["mol"] = train["SMILES"].apply(Chem.MolFromSmiles)

train[["SMILES", "mol"]].head()

,SMILES,mol
0,*CC(*)c1ccccc1C(=O)OCCCCCC,<rdkit.Chem.rdchem.Mol object at 0x0000019338D...
1,*Nc1ccc([C@H](CCC)c2ccc(C3(c4ccc([C@@H](CCC)c5...,<rdkit.Chem.rdchem.Mol object at 0x0000019338D...
2,*Oc1ccc(S(=O)(=O)c2ccc(Oc3ccc(C4(c5ccc(Oc6ccc(...,<rdkit.Chem.rdchem.Mol object at 0x0000019338D...
3,*Nc1ccc(-c2c(-c3ccc(C)cc3)c(-c3ccc(C)cc3)c(N*)...,<rdkit.Chem.rdchem.Mol object at 0x0000019338D...
4,*Oc1ccc(OC(=O)c2cc(OCCCCCCCCCOCC3CCCN3c3ccc([N...,<rdkit.Chem.rdchem.Mol object at 0x0000019338D...


In [17]:
# checking if there is a failed conversion from SMILES to mol objects by counting the number of NaN values in the "mol" column
train["mol"].isna().sum()

np.int64(0)

### Step2: Calculate molecular descriptors  


### Step 2.1: Define Molecular Descriptors

The following nine molecular descriptors are used to represent the structural and physicochemical properties of the polymers:

| Descripter | Bedeutung | Meaning (English) |
|---|---|---|
| `Molwt` | Die berechnete Molekülmasse auf Grundlage der chemischen Struktur. |Molecular weight calculated from the chemical structure. |
| `HeavyAtomCount` | Anzahl der Atome ohne Wasserstoffatome. | Number of atoms excluding hydrogen atoms. |
| `NumAtoms` | Gesamtzahl der Atome im RDKit-Molekül. | Total number of atoms in the RDKit molecule. |
| `NumRings` | Anzahl der Ringe in der Molekülstruktur. | Number of rings in the molecular structure. |
| `NumAromaticRings` | Anzahl der aromatischen Ringe in der Molekülstruktur. | Number of aromatic rings in the molecular structure. |
| `NumHDoners` | Anzahl der Wasserstoffbrücken-Donoren. | Number of hydrogen-bond donors. |
| `NumHAcceptors` | Anzahl der Wasserstoffbrücken-Akzeptoren. | Number of hydrogen-bond acceptors. |
| `MolLogP` | Berechneter Logarithmus des Octanol-Wasser-Verteilungskoeffizienten; ein Maß für die Lipophilie. | Calculated logarithm of the octanol–water partition coefficient; an indicator of lipophilicity. |
| `TPSA` | Topologische polare Oberfläche; ein berechnetes Maß für die polare Oberfläche eines Moleküls. | Topological polar surface area; a calculated measure of a molecule's polar surface area. |

### Step 2.2: Import required modules

In [ ]:
# Import modules for calculating molecular descriptors.
from rdkit.Chem import Descriptors
from rdkit.Chem import rdMolDescriptors

In [ ]:
# Select the first molecule as an example.

mol = train.loc[0, "mol"]

example_descriptors = {
    "MolWt": Descriptors.MolWt(mol),
    "HeavyAtomCount": Descriptors.HeavyAtomCount(mol),
    "NumAtoms": mol.GetNumAtoms(),
    "NumRings": rdMolDescriptors.CalcNumRings(mol),
    "NumAromaticRings": rdMolDescriptors.CalcNumAromaticRings(mol),
    "NumHDonors": rdMolDescriptors.CalcNumHBD(mol),
    "NumHAcceptors": rdMolDescriptors.CalcNumHBA(mol),
    "MolLogP": Descriptors.MolLogP(mol),
    "TPSA": Descriptors.TPSA(mol),
}

example_descriptors

{'MolWt': 232.3229999999999,
 'HeavyAtomCount': 17,
 'NumAtoms': 19,
 'NumRings': 1,
 'NumAromaticRings': 1,
 'NumHDonors': 0,
 'NumHAcceptors': 2,
 'MolLogP': 3.9817000000000036,
 'TPSA': 26.3}

### Step 2.3: Define a Function to Calculate Descriptors  
Define a function that calculates the nine molecular descriptors for each molecule. The function also handles invalid molecules by returning None.

In [24]:
# Define a function to calculate molecular descriptors.

def calculate_descriptors(mol):

    # Return None if the molecule is invalid.
    if mol is None:
        return None

    # Calculate and return nine molecular descriptors.
    return {
        "MolWt": Descriptors.MolWt(mol),
        "HeavyAtomCount": Descriptors.HeavyAtomCount(mol),
        "NumAtoms": mol.GetNumAtoms(),
        "NumRings": rdMolDescriptors.CalcNumRings(mol),
        "NumAromaticRings": rdMolDescriptors.CalcNumAromaticRings(mol),
        "NumHDonors": rdMolDescriptors.CalcNumHBD(mol),
        "NumHAcceptors": rdMolDescriptors.CalcNumHBA(mol),
        "MolLogP": Descriptors.MolLogP(mol),
        "TPSA": Descriptors.TPSA(mol),
    }

### Step 2.4: Define a Function to Calculate Descriptors  

Apply the descriptor calculation function to every molecule in the training dataset. Convert the results into a DataFrame to create a numerical feature matrix for machine learning.

In [25]:
# Calculate molecular descriptors for all molecules.
descriptor_data = train["mol"].apply(calculate_descriptors)

# Convert the descriptor results into a DataFrame.
descriptor_df = pd.DataFrame(descriptor_data.tolist(), index=train.index)

display(descriptor_df.head())

,MolWt,HeavyAtomCount,NumAtoms,NumRings,NumAromaticRings,NumHDonors,NumHAcceptors,MolLogP,TPSA
0,232.323,17,19,1,1,0,2,3.98170,26.30
1,598.919,45,47,5,4,2,2,12.35960,24.06
2,1003.207,73,75,10,8,0,9,14.21700,122.27
3,542.726,42,44,6,6,2,2,11.00768,24.06
4,965.154,70,72,6,4,0,14,11.84500,182.28


### Step 2.5: Verify the Feature Matrix  
Check the shape, missing values, and data types of the feature matrix to ensure that the molecular descriptors have been calculated successfully.

In [ ]:
# Check the shape (size) of the feature matrix.
print("Feature matrix shape:", descriptor_df.shape)

# Check for missing values in each descriptor.
print("\nMissing values per descriptor:")
print(descriptor_df.isna().sum())

# Check the data types of the descriptors.
print("\nData types:")
print(descriptor_df.dtypes)

Feature matrix shape: (7973, 9)

Missing values per descriptor:
MolWt               0
HeavyAtomCount      0
NumAtoms            0
NumRings            0
NumAromaticRings    0
NumHDonors          0
NumHAcceptors       0
MolLogP             0
TPSA                0
dtype: int64

Data types:
MolWt               float64
HeavyAtomCount        int64
NumAtoms              int64
NumRings              int64
NumAromaticRings      int64
NumHDonors            int64
NumHAcceptors         int64
MolLogP             float64
TPSA                float64
dtype: object


### Findings 
- Nine molecular descriptors were successfully calculated for all 7,973 polymer samples.  
- The resulting feature matrix contains 7,973 rows and 9 columns.  
- No missing values were found in any of the nine descriptors.  
- All descriptors have numerical data types (int64 or float64).  
- The feature matrix is ready for the next stage of feature preparation and   model development.